# GOBR — execução no Google Colab (GPU)

Este notebook **não substitui o projeto no Windows**. Ele baixa a mesma codebase do GitHub, instala as dependências, usa o Ollama **dentro da máquina do Colab** e executa a primeira entrega e a geração incremental de explicações (parser → interpretador → explicador).

**O que fica onde**
- **Windows:** edição da codebase e commits.
- **GitHub:** versão oficial do código para a disciplina.
- **Colab:** execução temporária com GPU (quando disponível).
- **Drive:** resultados das explicações.

O Colab gratuito tem limites variáveis de GPU/tempo e pode desconectar. **Não é um servidor permanente**. O Ollama não será exposto à internet.

**Antes de executar:** envie a pasta do projeto para um repositório no GitHub. Use um repositório público para o comando de clone mostrado aqui. Para repositório privado, não coloque tokens no código do notebook; use a alternativa de upload de ZIP ou configure autenticação de maneira segura.

## 1. Ative a GPU
No menu do Colab: **Ambiente de execução → Alterar o tipo de ambiente de execução → Acelerador de hardware: T4 GPU** (ou outra GPU disponível). Depois clique no botão ▶ desta célula.

A GPU deve aparecer em `nvidia-smi`. Se não aparecer, volte às configurações do ambiente e escolha uma GPU.

In [ ]:
import subprocess

subprocess.run(["nvidia-smi"], check=True)

## 2. Monte o Google Drive
O botão ▶ abrirá uma janela pedindo permissão para o Colab acessar seu Drive. As saídas serão salvas em `Meu Drive/gobr-agent/outputs/`. O código fica no GitHub, não no Drive.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 3. Baixe a mesma codebase do GitHub
Substitua a URL abaixo pela URL **real** do seu repositório público no GitHub. Exemplo: `https://github.com/usuario/sistemas-multiagentes.git`.

Sempre que você alterar o projeto no Windows e executar `git push`, execute novamente esta célula no Colab para baixar a nova versão. **Não edite agentes no Colab**, pois as alterações são temporárias e não voltam automaticamente ao GitHub.

**Alternativa:** se ainda não publicou seu repositório, altere `ORIGEM` para `"zip"`; o Colab solicitará o upload de um ZIP do projeto, sem enviar tokens de acesso.

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

ORIGEM = "github"  # "github" ou "zip"
REPO_URL = "https://github.com/SEU_USUARIO/SEU_REPOSITORIO.git"
PROJETO = Path('/content/gobr-project')

if 'src.demo_first_delivery' in sys.modules:
    raise RuntimeError(
        'O código antigo ainda está carregado na memória. '
        'Reinicie a sessão do Colab antes de clonar a nova revisão.'
    )

if PROJETO.exists():
    shutil.rmtree(PROJETO)  # Apenas a copia temporaria do Colab

if ORIGEM == "github":
    if "SEU_USUARIO" in REPO_URL or "SEU_REPOSITORIO" in REPO_URL:
        raise ValueError("Troque REPO_URL pela URL real do seu repositorio GitHub.")
    if not REPO_URL.startswith("https://github.com/"):
        raise ValueError("Use uma URL HTTPS de repositorio github.com.")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJETO)], check=True)
elif ORIGEM == "zip":
    from google.colab import files
    enviados = files.upload()
    if len(enviados) != 1:
        raise ValueError("Envie somente um arquivo ZIP da codebase.")
    nome_zip = next(iter(enviados))
    PROJETO.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(nome_zip) as pacote:
        for member in pacote.infolist():
            destino = (PROJETO / member.filename).resolve()
            if not destino.is_relative_to(PROJETO.resolve()):
                raise ValueError(f"Caminho inseguro no ZIP: {member.filename}")
        pacote.extractall(PROJETO)
else:
    raise ValueError("ORIGEM deve ser 'github' ou 'zip'.")

if not (PROJETO / 'src' / 'demo_first_delivery.py').is_file():
    raise FileNotFoundError("O projeto deve conter src/demo_first_delivery.py na raiz.")

os.chdir(PROJETO)
sys.path.insert(0, str(PROJETO))
print('Projeto carregado em:', PROJETO)
print('Arquivos de regras:', len(list((PROJETO / 'data' / 'referencias').glob('*.gobr'))))

## 4. Instale as dependências Python
Lê o mesmo `requirements.txt` versionado no GitHub. Esta instalação precisa ser refeita quando o Colab criar uma nova máquina virtual.

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJETO / 'requirements.txt')],
    check=True
)
print('Dependencias instaladas.')

## 5. Instale e inicie o Ollama **somente no Colab**
O Ollama executa o modelo dentro da máquina virtual do Colab. O servidor estará acessível **apenas pelo próprio notebook** em `127.0.0.1:11434`, não pelo Windows.

O script de instalação abaixo é o procedimento fornecido no site oficial do Ollama.

In [ ]:
subprocess.run(
    ['bash', '-lc', 'set -o pipefail; curl -fsSL https://ollama.com/install.sh | sh'],
    check=True
)

In [ ]:
import time
import urllib.request

os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
os.environ['OLLAMA_NUM_PARALLEL'] = '1'
os.environ['OLLAMA_MAX_LOADED_MODELS'] = '1'

def ollama_pronto():
    try:
        with urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

if not ollama_pronto():
    log = open('/tmp/ollama-gobr.log', 'a')
    processo_ollama = subprocess.Popen(
        ['ollama', 'serve'], stdout=log, stderr=subprocess.STDOUT,
        env=os.environ.copy()
    )

for _ in range(45):
    if ollama_pronto():
        break
    time.sleep(1)
else:
    raise RuntimeError('Ollama nao iniciou. Consulte /tmp/ollama-gobr.log')

print('Ollama pronto em http://127.0.0.1:11434')

## 6. Baixe o modelo maior
Começaremos com `gemma3:12b` (~8,2 GB de download de modelo). Ele deverá caber em muitas GPUs T4 de 16 GB, mas o uso real depende do contexto e da memória disponível. **Baixar o modelo não garante que a inferência use inteiramente a GPU.**

Se houver erro de memória, podemos testar um modelo menor. O modelo pode ser alterado aqui sem modificar os agentes.

In [ ]:
MODELO = 'gemma3:12b'
CONTEXTO = 4096

subprocess.run(['ollama', 'pull', MODELO], check=True)
os.environ['GOBR_LLM_MODEL'] = MODELO
os.environ['GOBR_LLM_NUM_CTX'] = str(CONTEXTO)
os.environ['OLLAMA_BASE_URL'] = 'http://127.0.0.1:11434'
print('Modelo preparado:', MODELO)

## 7. Execute **somente um exemplo**
Essa primeira execução valida o pipeline real: `GobrParser` → `GobrInterpreterAgent` → `BusinessExplainerAgent`. A saída será gravada no Google Drive.

A primeira resposta pode levar mais tempo para carregar o modelo na GPU.

In [ ]:
from src import demo_first_delivery as demo

TODOS_OS_CASOS = list(demo.CASES)
DESTINO = Path('/content/drive/MyDrive/gobr-agent/outputs/first_delivery_colab_v2')
DESTINO.mkdir(parents=True, exist_ok=True)

# O modulo original no GitHub NAO e alterado: apenas apontamos sua saida
# para o Drive e limitamos o primeiro teste a uma regra.
demo.OUTPUT_DIR = DESTINO
demo.CASES = TODOS_OS_CASOS[:1]
demo.main()

print('Verifique os arquivos no Drive:', DESTINO)

## 8. Confira se o Ollama realmente usou a GPU
O comando abaixo mostra os modelos carregados e o dispositivo usado. Se aparecer processamento misto GPU/CPU, pode significar memória de vídeo insuficiente para manter todo o modelo na GPU.

In [ ]:
subprocess.run(['ollama', 'ps'], check=True)

## 9. (Opcional) Execute os seis exemplos
Execute a célula **apenas depois** de verificar a explicação gerada na etapa 7. Alterar `EXECUTAR_SEIS` para `True` fará o notebook processar os seis exemplos da primeira entrega. As saídas serão gravadas na pasta `first_delivery_colab_v2` do Drive, sem sobrescrever a execução anterior.

**A preparação das regras e a geração em lote estão disponíveis nas novas etapas 11 a 15 abaixo.** A etapa 9 continua demonstrando apenas os seis casos iniciais.

In [ ]:
EXECUTAR_SEIS = False

if EXECUTAR_SEIS:
    demo.CASES = TODOS_OS_CASOS
    demo.main()
    print('Seis casos processados. Confira:', DESTINO)
else:
    print('Etapa opcional desativada. Para executar, mude EXECUTAR_SEIS para True.')

## 10. Registre a versão usada no experimento
O arquivo `run_metadata.json` no Drive associa os resultados ao modelo e, quando o código vier do GitHub, à revisão (commit) utilizada. Isso facilita documentar a disciplina.

In [ ]:
import datetime
import json

rev = None
if (PROJETO / '.git').exists():
    rev = subprocess.run(
        ['git', '-C', str(PROJETO), 'rev-parse', 'HEAD'],
        capture_output=True, text=True, check=True
    ).stdout.strip()

metadata = {
    'generated_at_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'model': MODELO,
    'num_ctx': CONTEXTO,
    'repository': REPO_URL if ORIGEM == 'github' else None,
    'commit': rev,
    'output_dir': str(DESTINO),
}
with (DESTINO / 'run_metadata.json').open('w', encoding='utf-8') as fp:
    json.dump(metadata, fp, indent=2, ensure_ascii=False)
print(json.dumps(metadata, indent=2, ensure_ascii=False))

## Quando você voltar amanhã
1. Se o código no GitHub mudou, reinicie a sessão do Colab (menu **Ambiente de execução → Reiniciar sessão**) antes de clonar, para evitar módulos Python antigos em memória.
2. Abra a versão atualizada deste notebook no GitHub.
3. Se necessário, ative a GPU e reconecte o Drive.
4. Execute as células de cima para baixo: a máquina virtual anterior pode ter sido apagada.
5. Os arquivos salvos no Drive continuam lá. O código original continua no GitHub.

Referências oficiais: [Google Colab — FAQ](https://research.google.com/colaboratory/intl/pt-BR/faq.html) e [Ollama — instalação Linux](https://ollama.com/download/linux).

## 11. Segunda entrega: preparar as 1.699 regras extraídas

Se você acabou de abrir o notebook nesta sessão, execute antes as **etapas 1 a 6** para ativar a GPU, carregar o GitHub e instalar o Gemma 12B. **Não precisa executar os seis exemplos novamente.**

Esta etapa só prepara o JSONL e verifica se o seu commit contém o executor. A saída temporária em `/content` será recriada a cada nova sessão.

In [ ]:
import sys
import subprocess
from pathlib import Path

if not (PROJETO / 'src' / 'batch_reference_explanations.py').is_file():
    raise FileNotFoundError('Atualize o repositório no GitHub, reinicie a sessão do Colab e clone novamente.')

subprocess.run([sys.executable, '-m', 'src.prepare_references'], cwd=str(PROJETO), check=True)
ENTRADA_REGRAS = PROJETO / 'outputs' / 'generated' / 'regras_extraidas.jsonl'
assert ENTRADA_REGRAS.is_file()

REVISAO_CODIGO = subprocess.run(
    ['git', '-C', str(PROJETO), 'rev-parse', '--short=8', 'HEAD'],
    capture_output=True, text=True
).stdout.strip() if (PROJETO / '.git').exists() else 'sem_git'

# Pasta independente por revisão do código e modelo.
NOME_MODELO = MODELO.replace(':', '_').replace('/', '_')
DESTINO_LOTE = Path('/content/drive/MyDrive/gobr-agent/outputs/reference_explanations') / f'{NOME_MODELO}_{REVISAO_CODIGO}'
DESTINO_LOTE.mkdir(parents=True, exist_ok=True)
print('Entrada:', ENTRADA_REGRAS)
print('Saída permanente no Drive:', DESTINO_LOTE)

## 12. Prévia do lote (sem executar o Gemma)

Verifique os IDs e nomes das primeiras regras. Esta célula não consome inferência.

In [ ]:
subprocess.run([
    sys.executable, '-m', 'src.batch_reference_explanations',
    '--input', str(ENTRADA_REGRAS),
    '--output-dir', str(DESTINO_LOTE),
    '--limit', '3', '--dry-run',
], cwd=str(PROJETO), check=True)

## 13. Primeiro lote: gerar somente três explicações

Cada regra gera um arquivo `items/<id>.json` no Drive assim que terminar. O sistema usa duas chamadas ao modelo por regra (interpretador + explicador). **Os textos ficam pendentes de revisão, não indexados.**

In [ ]:
subprocess.run([
    sys.executable, '-m', 'src.batch_reference_explanations',
    '--input', str(ENTRADA_REGRAS),
    '--output-dir', str(DESTINO_LOTE),
    '--limit', '3',
], cwd=str(PROJETO), check=True)

print('Abra esta pasta no Google Drive:', DESTINO_LOTE / 'items')

## 14. Examinar os resultados no próprio Colab

Confira `status`, `revisao.status`, `revisao.metodo` e um trecho da explicação. `completed` não significa aprovado: todos continuam com `indexable: false`.

In [ ]:
import json

for arquivo in sorted((DESTINO_LOTE / 'items').glob('*.json'))[:8]:
    item = json.loads(arquivo.read_text(encoding='utf-8'))
    explicacao = item.get('explanation') or {}
    revisao = explicacao.get('revisao') or {}
    print('\n---', item['id'], '|', item['status'], '|', revisao.get('status', '-'), '|', revisao.get('metodo', '-'))
    print((explicacao.get('explicacao') or item.get('reason') or str(item.get('error', '')))[:420])

## 15. Próximos lotes: retomada automática

Quando você estiver satisfeito com os 3 primeiros, ative `EXECUTAR_MAIS = True`. Na mesma pasta e mesmo commit, ele **pula as explicações já concluídas** e tenta até 20 novas regras. Execute a célula novamente para outro lote. Não use `--force` para retomar.

Não recomendo executar as 1.699 em uma única sessão gratuita: o Colab pode desconectar.

In [ ]:
EXECUTAR_MAIS = False

if EXECUTAR_MAIS:
    subprocess.run([
        sys.executable, '-m', 'src.batch_reference_explanations',
        '--input', str(ENTRADA_REGRAS),
        '--output-dir', str(DESTINO_LOTE),
        '--limit', '20',
    ], cwd=str(PROJETO), check=True)
else:
    print('Desativado. Altere EXECUTAR_MAIS para True depois de revisar o primeiro lote.')

### Ao retomar outro dia

1. Abra **este mesmo notebook**, ative a GPU, monte o Drive, clone o mesmo commit disponível no GitHub e inicie o Ollama (etapas 1 a 6).
2. Execute novamente as etapas **11** e **15**, com `EXECUTAR_MAIS = True`.
3. Use a mesma `DESTINO_LOTE`. Caso você tenha atualizado o GitHub desde a primeira rodada, será criada automaticamente **outra pasta por commit** para evitar misturar resultados antigos e novos.

Os JSONs já salvos no Drive não desaparecem com a desconexão do Colab. Os itens `failed` serão tentados novamente até o limite; `blocked_context` requer revisão de contexto.